#Projeto 2: Car Insurance Premium Prediction
O foco deste projeto é tratar os dados da base de dados contendo 1000 linhas de dados sintéticos que simulam prêmios de seguro de automóveis, calculados usando uma fórmula linear.

Será feito o tratamento tanto dos dados de treinamento, como também dos dados de teste.

###Pipeline

0. Importação dos dados
1. Uniformização dos nomes das colunas (lowercase, espaços em branco e caracteres especiais)
2. Tratamento dos outliers usando IQR e ZScore
3. Tratamento dos missing values
4. Lidar com dados categóricos
5. EDA
6. Descrever os tipos de dados

##0. Importando bibliotecas e dados

In [ ]:
#Importando bibliotecas necessárias
import numpy as np
import pandas as pd

In [ ]:
#Importando dados de treinamento (.csv) diretamente do GitHub
url_treinamento = 'https://raw.githubusercontent.com/MathMachado/DSWP/refs/heads/master/Projetos/Projeto2/car_insurance_premium_dataset.csv'
df_carIPP = pd.read_csv(url_treinamento)

#Criando cópia 'Silver' dos dados de treinamento
df_carIPP2 = df_carIPP.copy()
df_carIPP2.head()

,Driver Age,Driver Experience,Previous Accidents,Annual Mileage (x1000 km),Car Manufacturing Year,Car Age,Insurance Premium ($)
0,56,32,4,17,2002,23,488.35
1,46,19,0,21,2025,0,486.15
2,32,11,4,15,2020,5,497.55
3,60,0,4,19,1991,34,498.35
4,25,7,0,13,2005,20,495.55


In [ ]:
#Importando dados de teste (.csv)
url_teste = 'https://raw.githubusercontent.com/MathMachado/DSWP/refs/heads/master/Projetos/Projeto2/car_insurance_premium_dataset_TEST.csv'
df_carIPP_teste = pd.read_csv(url_teste)

#Criando cópia 'Silver' dos dados de teste
df_carIPP2_teste = df_carIPP_teste.copy()
df_carIPP2_teste.head()

,Driver Age,Driver Experience,Previous Accidents,Annual Mileage (x1000 km),Car Manufacturing Year,Car Age,Insurance Premium ($)
0,56,7,0,22,2009,16,489.40
1,46,23,2,21,1990,35,491.45
2,32,10,5,13,1997,28,501.55
3,60,16,1,16,2005,20,487.50
4,25,7,4,17,2003,22,501.95


##1. Uniformização dos nomes das colunas

###1.1 Aplicação do lowercase na tabela de treinamento e na de teste

In [ ]:
for col in df_carIPP2.columns:
  df_carIPP2 = df_carIPP2.rename(columns = {col : col.lower()})
  df_carIPP2_teste = df_carIPP2_teste.rename(columns = {col : col.lower()})

df_carIPP2.columns

Index(['driver age', 'driver experience', 'previous accidents',
       'annual mileage (x1000 km)', 'car manufacturing year', 'car age',
       'insurance premium ($)'],
      dtype='object')

##1.2 Remoção de caracteres especiais e espaços em branco


In [ ]:
#Convertendo colunas para lista e criando lista vazia
colunas = list(df_carIPP2.columns)
colunas_corrigidas = []

#Aplicando correção
for col in colunas:
  col = col.replace('(' , '')
  col = col.replace(')' , '')
  col = col.replace('$' , 's')
  col = col.replace(' ', '_')
  colunas_corrigidas.append(col)

#Substituindo nome das colunas pelo nome das colunas corrigido
#na tabela de treinamento e na de teste
df_carIPP2.columns = colunas_corrigidas
df_carIPP2_teste.columns = colunas_corrigidas
df_carIPP2.columns

Index(['driver_age', 'driver_experience', 'previous_accidents',
       'annual_mileage_x1000_km', 'car_manufacturing_year', 'car_age',
       'insurance_premium_s'],
      dtype='object')

##2. Tratamento dos outliers usando IQR e Z-Score


###2.1 Tratamento de outliers usando IQR

In [ ]:
#Cálculo do IQR e limites nos dados de treinamento

Q1 = df_carIPP2.quantile(0.25)
Q3 = df_carIPP2.quantile(0.75)
IQR = Q3 - Q1

Li = Q1 - 1.5 * IQR
print(f'Limites inferiores:\n{Li}\n')

Ls = Q3 + 1.5 * IQR
print(f'Limites superiores:\n{Ls}\n')

Limites inferiores:
driver_age                   -4.50
driver_experience           -19.50
previous_accidents           -3.50
annual_mileage_x1000_km       2.00
car_manufacturing_year     1972.00
car_age                     -19.00
insurance_premium_s         476.25
dtype: float64

Limites superiores:
driver_age                   87.50
driver_experience            48.50
previous_accidents            8.50
annual_mileage_x1000_km      34.00
car_manufacturing_year     2044.00
car_age                      53.00
insurance_premium_s         511.55
dtype: float64



In [ ]:
#Verificando a presença de outliers nos dados de treinamento
outliers_IQR = (df_carIPP2 < Li) | (df_carIPP2 > Ls)

df_carIPP2[outliers_IQR].sum()

,0
driver_age,0.0
driver_experience,0.0
previous_accidents,0.0
annual_mileage_x1000_km,0.0
car_manufacturing_year,0.0
car_age,0.0
insurance_premium_s,0.0


In [ ]:
#Cálculo do IQR e limites nos dados de teste

Q1_teste = df_carIPP2_teste.quantile(0.25)
Q3_teste = df_carIPP2_teste.quantile(0.75)
IQR_teste = Q3_teste - Q1_teste

Li_teste = Q1_teste - 1.5 * IQR_teste
print(f'Limites inferiores:\n{Li}\n')

Ls_teste = Q3_teste + 1.5 * IQR_teste
print(f'Limites superiores:\n{Ls}\n')

Limites inferiores:
driver_age                   -4.50
driver_experience           -19.50
previous_accidents           -3.50
annual_mileage_x1000_km       2.00
car_manufacturing_year     1972.00
car_age                     -19.00
insurance_premium_s         476.25
dtype: float64

Limites superiores:
driver_age                   87.50
driver_experience            48.50
previous_accidents            8.50
annual_mileage_x1000_km      34.00
car_manufacturing_year     2044.00
car_age                      53.00
insurance_premium_s         511.55
dtype: float64



In [ ]:
#Verificando a presença de outliers nos dados de teste
outliers_IQR_teste = (df_carIPP2_teste < Li_teste) | (df_carIPP2_teste > Ls_teste)
df_carIPP2_teste[outliers_IQR_teste].sum()


,0
driver_age,0.0
driver_experience,0.0
previous_accidents,0.0
annual_mileage_x1000_km,0.0
car_manufacturing_year,0.0
car_age,0.0
insurance_premium_s,0.0


###2.2 Tratamento dos outliers usando Z-Score

In [ ]:
#Calculando ZScore para dados de treinamento
media = df_carIPP2.mean()
dp = df_carIPP2.std()
ZScore = (df_carIPP2 - media) / dp

#Verificando a presença de outliers nos dados de treinamento
outliers_Z = ZScore.abs() > 3
ZScore[outliers_Z].sum()

,0
driver_age,0.0
driver_experience,0.0
previous_accidents,0.0
annual_mileage_x1000_km,0.0
car_manufacturing_year,0.0
car_age,0.0
insurance_premium_s,0.0


In [ ]:
#Calculando ZScore para dados de teste
media_teste = df_carIPP2_teste.mean()
dp_teste = df_carIPP2_teste.std()
ZScore_teste = (df_carIPP2_teste - media_teste) / dp_teste

#Verificando a presença de outliers nos dados de teste
outliers_Z_teste = ZScore_teste.abs() > 3
ZScore_teste[outliers_Z_teste].sum()

,0
driver_age,0.0
driver_experience,0.0
previous_accidents,0.0
annual_mileage_x1000_km,0.0
car_manufacturing_year,0.0
car_age,0.0
insurance_premium_s,0.0


###2.3 Conclusão

Nenhum outlier detectado tanto pelo método IQR quanto pelo Z-Score em nenhuma das bases de dados. Logo, não será necessário fazer a substituição dos outliers pela mediana, como sugerido.

##3. Tratamento dos missing values


In [ ]:
#Verificando a presença de missing values nos dados de treinamento
df_carIPP2.isna().sum()

,0
driver_age,0
driver_experience,0
previous_accidents,0
annual_mileage_x1000_km,0
car_manufacturing_year,0
car_age,0
insurance_premium_s,0


In [ ]:
#Verificando a presença de missing values nos dados de teste
df_carIPP2_teste.isna().sum()

,0
driver_age,0
driver_experience,0
previous_accidents,0
annual_mileage_x1000_km,0
car_manufacturing_year,0
car_age,0
insurance_premium_s,0


Observou-se que não há missing values, logo, não é necessário fazer a substituição pela mediana, como sugerido.

##4. Lidar com dados categóricos

In [ ]:
#Descobrir quais variáveis são categóricas
df_carIPP2.dtypes


,0
driver_age,int64
driver_experience,int64
previous_accidents,int64
annual_mileage_x1000_km,int64
car_manufacturing_year,int64
car_age,int64
insurance_premium_s,float64


Conforme observado, todos os dados são 'int64' ou 'float64'. Como dados categóricos são strings, então não há dados categóricos na base de dados df_carIPP2. Caso houvesse, poderia ser feito a recodigicação dos dados de forma ordenada por números naturais.

##5. EDA

In [ ]:
#Contagem, média, desvio padrão, mínimo, p25, p50, p75 e máximo dos dados de treinamento
df_carIPP2.describe()

,driver_age,driver_experience,previous_accidents,annual_mileage_x1000_km,car_manufacturing_year,car_age,insurance_premium_s
count,1000.000000,1000.000000,1000.0000,1000.000000,1000.000000,1000.000000,1000.000000
mean,41.575000,14.759000,2.5680,17.933000,2007.637000,17.363000,493.742250
std,13.765677,10.544292,1.6989,4.410665,10.363331,10.363331,5.909689
min,18.000000,0.000000,0.0000,11.000000,1990.000000,0.000000,477.050000
25%,30.000000,6.000000,1.0000,14.000000,1999.000000,8.000000,489.487500
50%,42.000000,13.000000,3.0000,18.000000,2008.000000,17.000000,493.950000
75%,53.000000,23.000000,4.0000,22.000000,2017.000000,26.000000,498.312500
max,65.000000,40.000000,5.0000,25.000000,2025.000000,35.000000,508.150000


In [ ]:
#Contagem, média, desvio padrão, mínimo, p25, p50, p75 e máximo dos dados de teste
df_carIPP2_teste.describe()

,driver_age,driver_experience,previous_accidents,annual_mileage_x1000_km,car_manufacturing_year,car_age,insurance_premium_s
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
mean,40.920000,13.660000,2.240000,17.920000,2008.000000,17.000000,493.674000
std,14.054497,10.318386,1.700386,4.469154,10.771267,10.771267,5.757233
min,18.000000,0.000000,0.000000,11.000000,1990.000000,0.000000,481.550000
25%,30.500000,5.000000,1.000000,14.000000,1999.000000,7.750000,489.737500
50%,41.000000,11.500000,2.000000,18.000000,2009.000000,16.000000,494.200000
75%,53.250000,23.000000,3.000000,22.000000,2017.250000,26.000000,497.800000
max,65.000000,38.000000,5.000000,25.000000,2025.000000,35.000000,508.300000


Com base nas medidas de posição que foram apresentadas acima, observou-se que a média e a mediana de todas as variáveis estão bem próximas, o que indica que os dados são aproximadamente simétricos e, por consequência, facilitará a modelagem.

In [ ]:
#Cálculo do coeficiente de variação dos dados de treinamento
100 * df_carIPP2.std() / df_carIPP2.mean()

,0
driver_age,33.110468
driver_experience,71.443135
previous_accidents,66.156554
annual_mileage_x1000_km,24.595243
car_manufacturing_year,0.516195
car_age,59.686293
insurance_premium_s,1.196918


In [ ]:
#Cálculo do coeficiente de variação dos dados de teste
100 * df_carIPP2_teste.std() / df_carIPP2_teste.mean()

,0
driver_age,34.346278
driver_experience,75.537233
previous_accidents,75.910097
annual_mileage_x1000_km,24.939473
car_manufacturing_year,0.536418
car_age,63.360397
insurance_premium_s,1.166201


O coeficiente de variação pode ser interpretado da seguinte forma:
- CV < 15: Baixa dispersão e dados mais próximos da média
- 15 < CV < 30: Dispersão média
- CV > 30: Alta dispersão e dados mais espalhados em torno da média

Percebeu-se que, tanto as variáveis 'car_manufacturing_year' como 'insurance_premium_s' possuem baixíssima dispersão, o que implica em prêmios homogêneos.

Já as demais possuem dispersão moderada ou alta, com destaque para 'driver_experience' e 'previus_accidents' sendo mais altas. Isso indica que o tempo de experiência dos motoristas é bem variado, assim como a quantidade de acidentes anteriores, resultando em diversos perfis de risco.

##6. Descrever os tipos de dados

In [ ]:
#Verificação dos dados
df_carIPP2.dtypes

,0
driver_age,int64
driver_experience,int64
previous_accidents,int64
annual_mileage_x1000_km,int64
car_manufacturing_year,int64
car_age,int64
insurance_premium_s,float64


Os dados 'int64' são inteiros e positivos, usando 64 bits de memória por valor, enquanto os dados 'float64' são números reais, que também usam 64 bits de memória por valor.

Alguns outros tipos de dados que não estão disponíveis nas bases de dados trabalhadas são:

- string: Dados textuais
- bool: Podem ser ou verdadeiros ou falsos
- datatime: Data e hora